# Errors, Exceptions, Warnings and Logging in Python

**Goal of this lesson:** understand what happens when things go wrong in a program, and how to deal with it like a professional.

By the end you will know how to:
1. Read an error message (a *traceback*) without panicking
2. Recognise the most common Python errors
3. Catch errors with `try` / `except` / `else` / `finally`
4. Create your own errors with `raise`
5. Use **warnings** for "this is suspicious, but I can continue"
6. Use **logging** instead of `print` to keep track of what your program does

---
## 1. Two kinds of errors

### 1.1 Syntax errors: "I can't even read this"
A **syntax error** happens when the code breaks the grammar rules of Python, like writing a sentence with no verb. Python refuses to run *anything* in the cell.

Run the cell below. Notice that the first `print` is never executed, because Python reads the whole cell before running it.

In [1]:
print("This line will NOT run")   # code is fine here, but there's a syntax error in the cell
print("Hello"     # missing closing parenthesis

_IncompleteInputError: incomplete input (1289819585.py, line 2)

### 1.2 Exceptions: "I understood you, but I can't do it"
An **exception** happens while the program is running. The syntax is fine, but the action is impossible.

This time the first line **does** run, and the program stops at the line that fails.

In [2]:
print("This line runs fine")
result = 10 / 0
print("This line is never reached")

This line runs fine


ZeroDivisionError: division by zero

---
## 2. How to read a traceback

When an exception happens, Python prints a **traceback**. It looks scary, but it follows a fixed structure.

**Read it from the BOTTOM up:**
1. **Last line** → the *type* of error and a message (e.g. `ZeroDivisionError: division by zero`)
2. **Lines above** → *where* it happened (file, line number, the code)
3. **Further up** → the chain of function calls that led there

Let's see a traceback that goes through several functions:

In [5]:
def compute_average(numbers):
    return sum(numbers) / len(numbers)

def report(numbers):
    avg = compute_average(numbers)
    print("The average is", avg)

number = []
report(number)   # an empty list!

ZeroDivisionError: division by zero

**Reading it:**
- Bottom line: `ZeroDivisionError: division by zero` → *what* happened
- Just above: the line `return sum(numbers) / len(numbers)` → *where* it happened
- Above that: `report([])` called `compute_average` → *how we got there*

The real cause is that we passed an empty list, so `len(numbers)` is `0`.

---
## 3. The most common exceptions

You will meet these over and over. Run each cell, read the last line of the traceback, and see how the name of the error already describes the problem.

| Exception | Meaning in plain words |
|---|---|
| `ZeroDivisionError` | You divided by zero |
| `ValueError` | Right type, but the value makes no sense (`int("hello")`) |
| `TypeError` | Wrong type for the operation (`"5" + 3`) |
| `NameError` | You used a variable that doesn't exist (often a typo) |
| `IndexError` | List position doesn't exist |
| `KeyError` | Dictionary key doesn't exist |
| `AttributeError` | The object doesn't have that method or attribute |
| `FileNotFoundError` | The file you tried to open isn't there |

In [6]:
# ValueError: the type is right (a string), but its content can't become a number
int("hello")

ValueError: invalid literal for int() with base 10: 'hello'

In [7]:
# TypeError: you can't add text and a number
"Age: " + 30

TypeError: can only concatenate str (not "int") to str

In [8]:
# NameError: typo in the variable name
student_name = "Anna"
print(studnet_name)

NameError: name 'studnet_name' is not defined

In [9]:
# IndexError: the list has 3 elements, at positions 0, 1, 2
grades = [28, 30, 25]
grades[3]

IndexError: list index out of range

In [10]:
# KeyError: the key does not exist in the dictionary
ages = {"Anna": 21, "Marco": 23}
ages["Luca"]

KeyError: 'Luca'

In [11]:
# AttributeError: lists don't have a method called 'push' (it's 'append')
grades = [28, 30]
grades.push(25)

AttributeError: 'list' object has no attribute 'push'

In [12]:
# FileNotFoundError: the file doesn't exist
open("file_that_does_not_exist.txt")

FileNotFoundError: [Errno 2] No such file or directory: 'file_that_does_not_exist.txt'

💡 **All these errors are related.** They are all "children" of a general class called `Exception`. This matters later: catching `Exception` catches almost everything, which is usually *too much*.

---
## 4. Catching exceptions: `try` / `except`

So far every error **stopped the program**. Often we want to *handle* the problem instead, for example by asking the user again or using a default value.

```python
try:
    # code that might fail
except SomeError:
    # what to do if it fails
```


In [13]:
user_input = "abc"   # imagine the user typed this

try:
    age = int(user_input)
    print("You are", age, "years old")
except ValueError:                                # if try fails, except will be called, otherwise it will be skipped
    print("That's not a valid number!")

print("The program continues normally.")

That's not a valid number!
The program continues normally.


### 4.1 Getting the error message with `as`
`except ValueError as e` stores the error in the variable `e`, so you can show or record the message.

In [ ]:
try:
    int("abc")
except ValueError as e:
    print("Something went wrong:", e)
    print("Error type:", type(e).__name__)     # check the type of the exception

Something went wrong: invalid literal for int() with base 10: 'abc'
Error type: ValueError


### 4.2 Handling different errors differently
You can have several `except` blocks. Python checks them **in order** and runs the first one that matches.

In [16]:
def safe_divide(a, b):
    try:
        result = a / b
        print(f"{a} / {b} = {result}")
    except ZeroDivisionError:
        print("ZeroDivisionError: You can't divide by zero.")
    except TypeError:
        print("TypeError: Both values must be numbers.")

safe_divide(10, 2)   # fine
safe_divide(10, 0)   # division by zero
safe_divide(10, "two")  # type error

10 / 2 = 5.0
ZeroDivisionError: You can't divide by zero.
TypeError: Both values must be numbers.


### 4.3 ⚠️ Don't catch everything blindly
This looks convenient but is a **bad habit**:
```python
try:
    ...
except:          # catches EVERYTHING, even typos!
    pass         # and silently ignores them
```
Look what happens when we make a typo inside the `try`:

In [17]:
numbers = [1, 2, 3]

try:
    total = sum(nubmers)        # typo! NameError
except:
    print("The list is empty!")  # wrong explanation, the real bug is hidden

The list is empty!


**Rule of thumb:** catch only the errors you *expect* and *know how to handle*. Let everything else crash loudly so you can find and fix it.

---
## 5. `else` and `finally`

The complete structure is:
```python
try:
    # code that might fail
except SomeError:
    # runs ONLY if that error happened
else:
    # runs ONLY if NO error happened
finally:
    # runs ALWAYS, error or not
```

`finally` is the place for *cleanup*: closing files, closing connections, releasing resources.

In [18]:
def convert(text):
    print(f"--- converting {text!r} ---")
    try:
        number = int(text)
    except ValueError:
        print("except:  conversion failed")
    else:
        print("else:    conversion worked, number =", number)
    finally:
        print("finally: this always runs")

convert("42")
convert("forty-two")

--- converting '42' ---
else:    conversion worked, number = 42
finally: this always runs
--- converting 'forty-two' ---
except:  conversion failed
finally: this always runs


### 5.1 `finally` runs even if the error is NOT caught
Here the error is a `ZeroDivisionError`, but we only catch `ValueError`. The program crashes, but `finally` still runs first.

In [23]:
try:
    print("Opening connection...")
    x = 1 / 0
except ValueError:
    print("This won't run")
    print("Neither will this")
finally:
    print("Closing connection (always!)")

Opening connection...
Closing connection (always!)


ZeroDivisionError: division by zero

### 5.2 A real-world case: files
Opening a file that may not exist is a classic case for `try`/`except`.

(In everyday code you'll use `with open(...)`, which closes the file automatically, effectively a built-in `finally`.)

In [26]:
# First, let's create a small file to work with
with open("grades.txt", "w") as f:
    f.write("28\n30\nabsent\n25\n")

def read_grades(filename):
    try:
        with open(filename) as f:
            lines = f.read().splitlines()
    except FileNotFoundError:
        print(f"File {filename!r} not found.")
        return []

    grades = []
    for line in lines:
        try:
            grades.append(int(line))
        except ValueError:
            print(f"Skipping invalid line: {line!r}")
    return grades

print(read_grades("grades.txt"))
print(read_grades("missing.txt"))

Skipping invalid line: 'absent'
[28, 30, 25]
File 'missing.txt' not found.
[]


---
## 6. Raising your own errors with `raise`

So far Python raised the errors. You can raise them **yourself** when your code receives something that makes no sense.

In [27]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"Age must be an integer, got {type(age).__name__}")
    if age < 0:
        raise ValueError(f"Age cannot be negative, got {age}")
    print("Age set to", age)

set_age(25)

Age set to 25


In [28]:
set_age(-5)

ValueError: Age cannot be negative, got -5

And because `set_age` raises standard exceptions, whoever *uses* the function can catch them with `try`/`except`:

In [29]:
for value in [30, -2, "twenty"]:
    try:
        set_age(value)
    except (ValueError, TypeError) as e:   # catch two types at once
        print("Invalid input:", e)

Age set to 30
Invalid input: Age cannot be negative, got -2
Invalid input: Age must be an integer, got str


### 6.1 Custom exceptions
For your own projects you can create **your own exception types**. It's as simple as creating a class that inherits from `Exception`. This makes errors more meaningful and easier to catch specifically.

In [31]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal is larger than the balance."""
    pass

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(
            f"Cannot withdraw {amount}€, balance is only {balance}€"
        )
    return balance - amount

try:
    new_balance = withdraw(100, 250)
except InsufficientFundsError as e:
    print("Transaction refused:", e)

Transaction refused: Cannot withdraw 250€, balance is only 100€


---
## 7. Warnings: "Careful, but I'll continue"

Not every problem should stop the program. A **warning** says: *"this looks suspicious, you should check, but I can go on."*

Common uses:
- A function is **deprecated** (will be removed in a future version)
- Data looks strange but is still usable
- A result might be imprecise

In [32]:
import warnings

def average(numbers):
    if len(numbers) < 3:
        warnings.warn("Very few values: the average may not be meaningful")
    return sum(numbers) / len(numbers)

result = average([10, 20])
print("Result:", result)    # the program continues!

Result: 15.0


/tmp/ipykernel_109218/1807829742.py:5: UserWarning: Very few values: the average may not be meaningful
  warnings.warn("Very few values: the average may not be meaningful")


### 7.1 Warning categories
Like exceptions, warnings have types. The most common:

| Category | When to use it |
|---|---|
| `UserWarning` | Default, generic warning |
| `DeprecationWarning` | "This will be removed in the future, use something else" |
| `RuntimeWarning` | Something dubious happened while running |
| `FutureWarning` | "This behaviour will change in a future version" |

You'll see `FutureWarning` and `DeprecationWarning` often from libraries like **pandas** and **numpy**.

In [38]:
def old_function():                           # The function does nothing and return None. Just to show how deprecation warnings are used.
    warnings.warn(
        "old_function() is deprecated, use new_function() instead",
        DeprecationWarning
    )
    return None

old_function()

/tmp/ipykernel_109218/2739956135.py:2: DeprecationWarning: old_function() is deprecated, use new_function() instead
  warnings.warn(


### 7.2 Controlling warnings
You can **hide** warnings you already know about, or turn them **into errors** (useful while testing, to make sure nothing suspicious slips through).

The `with warnings.catch_warnings():` block keeps these changes temporary.

In [45]:
# Hide warnings (only inside this block)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")      # ignore let us hide the warning message
    print("Silent:", average([1, 2]))
    print("I can still print this line, even if the average() function issues a warning.")

# Turn warnings into errors (only inside this block)
with warnings.catch_warnings():
    warnings.simplefilter("error")       # error converts warnings into exceptions
    average([1, 2])
    print("This line will not be printed!")

Silent: 1.5
I can still print this line, even if the average() function issues a warning.


UserWarning: Very few values: the average may not be meaningful

In [46]:
# CORRECT WAY OF HANDLING ERRROS: Turn warnings into errors (only inside this block)
with warnings.catch_warnings():
    warnings.simplefilter("error")       # error converts warnings into exceptions
    try:
        average([1, 2])
    except UserWarning as w:
        print("The warning became an error:", w)
        print("This line will still be printed")

The warning became an error: Very few values: the average may not be meaningful
This line will still be printed


⚠️ Ignoring warnings is fine when you **understand** them. Ignoring them just because they are annoying is like covering the "low fuel" light with tape.

---
## 8. Logging: a diary for your program

Until now we used `print` to see what's happening. That works for small scripts, but in real programs `print` has problems:
- You can't easily turn messages **on/off**
- You can't distinguish **important** messages from **details**
- There's no **timestamp**, so you don't know *when* something happened
- Messages only go to the screen, not to a **file** you can read later

The `logging` module solves all of this.

### 8.1 The five levels
Every message has a **level** that says how important it is:

| Level | When to use it | Example |
|---|---|---|
| `DEBUG` | Detailed info, useful only when hunting bugs | "Variable x = 42" |
| `INFO` | Normal confirmation that things work | "File loaded successfully" |
| `WARNING` | Something unexpected, but the program works | "Disk almost full" |
| `ERROR` | Something failed | "Could not save the file" |
| `CRITICAL` | Serious failure, the program may not continue | "Database unreachable" |

You choose a **minimum level**: messages below it are hidden.

In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.DEBUG,                                   # show everything from DEBUG up
    format="%(asctime)s | %(levelname)-8s | %(message)s",  # what each line looks like
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,   # needed in Jupyter to apply the configuration again
)

logging.debug("Detail for developers")
logging.info("Everything is working")
logging.warning("Something looks odd")
logging.error("Something failed")
logging.critical("Big trouble!")

Now let's raise the minimum level to `WARNING`. The `DEBUG` and `INFO` messages disappear **without touching the rest of the code**. That's something `print` can't do.

In [ ]:
logging.getLogger().setLevel(logging.WARNING)

logging.debug("You won't see me")
logging.info("You won't see me either")
logging.warning("But you will see me")

logging.getLogger().setLevel(logging.DEBUG)   # back to showing everything

### 8.2 Named loggers
In real projects each file or module creates its **own logger** with a name. That way you can see *which part* of the program wrote each message.

The standard pattern is:
```python
logger = logging.getLogger(__name__)
```
Here we use explicit names to make it visible:

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(name)-10s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)

db_logger = logging.getLogger("database")
ui_logger = logging.getLogger("interface")

db_logger.info("Connected")
ui_logger.info("Window opened")
db_logger.warning("Query was slow (3.2 s)")

### 8.3 Logging exceptions
Inside an `except` block, use `logger.exception(...)`. It logs at `ERROR` level **and** includes the full traceback, so you don't lose information about what went wrong, but the program keeps running.

In [ ]:
logger = logging.getLogger("calculator")

def divide(a, b):
    logger.debug(f"divide called with a={a}, b={b}")
    try:
        result = a / b
    except ZeroDivisionError:
        logger.exception("Division failed")
        return None
    logger.info(f"Result: {result}")
    return result

divide(10, 2)
divide(10, 0)
print("The program is still alive!")

### 8.4 Writing logs to a file
Logs are most useful when saved to a **file**, so you can read them after the program has finished, even days later.

In [ ]:
file_logger = logging.getLogger("file_demo")
file_logger.setLevel(logging.INFO)
file_logger.propagate = False   # don't also print to the screen

handler = logging.FileHandler("app.log", mode="w")
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
file_logger.addHandler(handler)

file_logger.info("Program started")
file_logger.warning("Configuration file missing, using defaults")
file_logger.error("Could not reach the server")
handler.close()

# Read the file back
with open("app.log") as f:
    print(f.read())

---
## 9. Putting it all together

A small program that reads student grades. It uses **everything** we've seen:
- `raise` to validate data
- a **custom exception**
- `try` / `except` / `else` / `finally`
- a **warning** for suspicious data
- **logging** to record what happens

In [ ]:
import logging, sys, warnings

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)-8s | %(message)s",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("grades")


class InvalidGradeError(Exception):
    """Raised when a grade is outside the allowed range."""


def parse_grade(text):
    grade = int(text)                 # may raise ValueError
    if not 18 <= grade <= 30:
        raise InvalidGradeError(f"grade {grade} is not between 18 and 30")
    return grade


def process(raw_grades):
    logger.info(f"Processing {len(raw_grades)} entries")
    valid = []
    try:
        for raw in raw_grades:
            try:
                g = parse_grade(raw)
            except ValueError:
                logger.warning(f"Not a number, skipped: {raw!r}")
            except InvalidGradeError as e:
                logger.error(f"Invalid grade, skipped: {e}")
            else:
                valid.append(g)

        if len(valid) < 3:
            warnings.warn("Fewer than 3 valid grades: average is unreliable")

        avg = sum(valid) / len(valid)  # may raise ZeroDivisionError
    except ZeroDivisionError:
        logger.critical("No valid grades at all, cannot compute an average")
        return None
    finally:
        logger.info("Processing finished")

    logger.info(f"Average: {avg:.2f}")
    return avg


process(["28", "30", "abc", "45", "25", "27"])

In [ ]:
process(["hello", "99"])

---
## 10. Summary

| Tool | What it does | When to use it |
|---|---|---|
| **Traceback** | Shows what went wrong and where | Read it bottom-up, every time |
| `try` / `except` | Catches an error and handles it | When you *expect* an error and know what to do |
| `else` | Runs if no error occurred | Code that should run only on success |
| `finally` | Always runs | Cleanup (close files, connections) |
| `raise` | Creates an error | Validate inputs, fail early |
| `warnings.warn` | Signals a problem without stopping | Suspicious data, deprecated features |
| `logging` | Records what happens, with levels | Instead of `print` in real programs |

### Golden rules
1. ✅ Catch **specific** exceptions (`except ValueError`), not everything
2. ❌ Never use a bare `except:` followed by `pass`
3. ✅ Fail early with a **clear message** (`raise ValueError("...")`)
4. ✅ Use `finally` (or `with`) for cleanup
5. ✅ Don't ignore warnings you don't understand
6. ✅ Use `logging` instead of `print` for anything beyond quick tests

---
## 11. Exercises

**Exercise 1.** Write a function `ask_number(text)` that converts `text` to an `int`. If the conversion fails, print a friendly message and return `0`.

**Exercise 2.** Write a function `get_item(lst, index)` that returns `lst[index]`, but returns `None` and logs a warning if the index doesn't exist.

**Exercise 3.** Write a function `set_temperature(celsius)` that raises a `ValueError` if the temperature is below -273.15 (absolute zero), and issues a **warning** if it's above 100.

**Exercise 4.** Find the bug: why does this code say "File not found" even though the problem is something else?
```python
try:
    with open("grades.txt") as f:
        data = f.read()
    total = sum(dat)
except:
    print("File not found")
```

In [ ]:
# Write your solutions here

---
### Solutions
(Try first! Then run the cell below.)

In [ ]:
import logging, warnings

# Exercise 1
def ask_number(text):
    try:
        return int(text)
    except ValueError:
        print(f"{text!r} is not a number, using 0")
        return 0

print(ask_number("12"), ask_number("twelve"))

# Exercise 2
def get_item(lst, index):
    try:
        return lst[index]
    except IndexError:
        logging.warning(f"Index {index} out of range (list has {len(lst)} items)")
        return None

print(get_item([1, 2, 3], 1), get_item([1, 2, 3], 10))

# Exercise 3
def set_temperature(celsius):
    if celsius < -273.15:
        raise ValueError(f"{celsius}°C is below absolute zero")
    if celsius > 100:
        warnings.warn(f"{celsius}°C is above boiling point, are you sure?")
    return celsius

set_temperature(150)
try:
    set_temperature(-300)
except ValueError as e:
    print("Error:", e)

# Exercise 4
# The bug is a typo: 'dat' instead of 'data' (NameError).
# The bare 'except:' catches it and prints a WRONG explanation.
# Fix: catch only FileNotFoundError, so the NameError shows up and can be fixed.